# DP2 — View coaads from Static Objects stracts 

**Author:** dagoret  
**Creation Date:** 2026-03-19  
**Last UpDate:** 2026-03-19 : add selection Isolated Star, (not extended and primary)   <==> drop patch selection
**version:** v1
## Purpose

Retrieve **Object**, **Source**, and **ForcedSource** catalogs
for a user-selected Deep Drilling Field (DDF) using **Butler Gen3 only**
(no TAP service, not yet available for DP2 at USDF).

## Actual schema (discovered at runtime — COSMOS, LSSTCam/runs/DRP/20250417_20250921/w_2025_49/DM-53545)

- `object` : dims `(skymap, tract)` — 1 ref per tract, index = `ObjectId`
- `source` : dims `(skymap, tract)` — 1 ref per tract, index = `SourceId` (join on `ObjectId` column)
- `object_forced_source` : dims `(skymap, tract, patch)` — **21 refs per tract** (one per patch)
  - Must iterate over all patch refs and concat
  - Contains per-visit forced photometry linked to DiaObjects

## Reference notebooks
- `2026-03-07_AccessToDP2.ipynb` — Butler setup
- `2026-03-13_DP2_DDF_Tracts_SkyMap_Mollweide.ipynb` — tract lookup

---
## 0. Imports

In [ ]:
import warnings

warnings.filterwarnings("ignore")
import traceback
import gc


import os

import numpy as np
import pandas as pd
import matplotlib
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from astropy.coordinates import SkyCoord
from astropy.table import Table
import astropy.units as u
from astropy.time import Time
from astropy.coordinates import Angle

from scipy.stats import median_abs_deviation

from lsst.daf.butler import Butler
from lsst.geom import SpherePoint, degrees

# Firefly : lequel utiliser ?
from firefly_client import FireflyClient
import lsst.afw.display as afwDisplay

print(f"Matplotlib : {matplotlib.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"Pandas     : {pd.__version__}")
print("Imports OK")

In [ ]:
mpl.rcParams.update(
    {
        "figure.figsize": (8, 5),  # default figure size
        "font.size": 14,  # taille de base
        "axes.titlesize": 18,  # titre de l'axe
        "axes.labelsize": 16,  # labels x et y
        "xtick.labelsize": 14,  # ticks x
        "ytick.labelsize": 14,  # ticks y
        "legend.fontsize": 14,  # legend text
        "legend.title_fontsize": 15,  # legend title
        "figure.titlesize": 20,  # titre global de la figure
    }
)

In [ ]:
# Remove to run faster the notebook
# import ipywidgets as widgets
# %matplotlib widget

# Enable interactive matplotlib backend with zoom/pan toolbar
# Requires: pip install ipympl
# If ipympl is not available, fall back to inline (no interactivity)
# try:
#    import ipympl  # noqa: F401
#    %matplotlib widget
#    print("ipympl found → interactive backend (%matplotlib widget)")
# except ImportError:
#    %matplotlib inline
#    print("ipympl NOT found → falling back to %matplotlib inline (no zoom widget)")
#    print("Install with:  pip install ipympl")


In [ ]:
# output dirs
NB_TAG = "DP2_DDF_COADDS_BUTLER_01"
DIR_DATA = f"data_{NB_TAG}"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_DATA, exist_ok=True)
os.makedirs(DIR_FIGS, exist_ok=True)
print(f"Data : {os.path.abspath(DIR_DATA)}")
print(f"Figs : {os.path.abspath(DIR_FIGS)}")


# Output directory for DRP data
OUTPUT_DIR = DIR_DATA
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Visits filename
FILENAME_VISITS = "visitTable-2025041500138-2026010800515_N34276.csv"

In [ ]:
# debug flags
DEBUG = False
DEBUG_VISITS = False

In [ ]:
# FLAGS
FLAG_FETCH_VISITSEXPOSURES = False

## Usefull functions

- from https://github.com/sylvielsstfr/mySIT-COM2025/blob/main/notebooks/2025-05-30-Visits/05_SourcesFromVisits.ipynb

In [ ]:
def getLostOfVisits(registry, where_clause_date):
    """
    Get exposure/visits from butler registry
    parameters:
    ==========
        registry : butler registry
        where_clause_date : specify which instrument and exposure dates in the butler registry
    """
    df_exposure = pd.DataFrame(
        {
            "id": pd.Series(dtype="int"),
            "obs_id": pd.Series(dtype="int"),
            "day_obs": pd.Series(dtype="int"),
            "seq_num": pd.Series(dtype="int"),
            "time_start": pd.Series(dtype="str"),  # ou 'datetime64[ns]' si c'est un datetime
            "time_end": pd.Series(dtype="str"),  # idem
            "type": pd.Series(dtype="str"),
            "target": pd.Series(dtype="str"),
            "filter": pd.Series(dtype="str"),
            "zenith_angle": pd.Series(dtype="float"),
            "expos": pd.Series(dtype="float"),  # ou 'int' selon le cas
            "ra": pd.Series(dtype="float"),
            "dec": pd.Series(dtype="float"),
            "skyangle": pd.Series(dtype="float"),
            "azimuth": pd.Series(dtype="float"),
            "zenith": pd.Series(dtype="float"),
            "science_program": pd.Series(dtype="str"),
            "jd": pd.Series(dtype="float"),
            "mjd": pd.Series(dtype="float"),
        }
    )

    # save the data array in rows before saving in pandas dataframe
    rows = []
    for count, info in enumerate(registry.queryDimensionRecords("exposure", where=where_clause_date)):
        try:
            jd_start = info.timespan.begin.value
            jd_end = info.timespan.end.value
            the_Time_start = Time(jd_start, format="jd", scale="utc")
            the_Time_end = Time(jd_end, format="jd", scale="utc")
            mjd_start = the_Time_start.mjd
            mjd_end = the_Time_end.mjd
            isot_start = the_Time_start.isot
            isot_end = the_Time_end.isot

            if count == 0:
                print("===== Time Conversion Debug Info =====")
                print(f"JD start      : {jd_start} (type: {type(jd_start)})")
                print(f"JD end        : {jd_end} (type: {type(jd_end)})")
                print(f"MJD start     : {mjd_start} (type: {type(mjd_start)})")
                print(f"MJD end       : {mjd_end} (type: {type(mjd_end)})")
                print(f"ISOT start    : {isot_start} (type: {type(isot_start)})")
                print(f"ISOT end      : {isot_end} (type: {type(isot_end)})")
                print("=======================================")

            # put row in a dictionnary before stacking
            row = {
                "id": info.id,
                "obs_id": info.obs_id,
                "day_obs": info.day_obs,
                "seq_num": info.seq_num,
                "time_start": isot_start,
                "time_end": isot_end,
                "type": info.observation_type,
                "target": info.target_name,
                "filter": info.physical_filter,
                "zenith_angle": info.zenith_angle,
                "expos": info.exposure_time,  # Exemple : adapter selon ton objet
                "ra": info.tracking_ra,
                "dec": info.tracking_dec,
                "skyangle": info.sky_angle,
                "azimuth": info.azimuth,
                "zenith": info.zenith_angle,
                "science_program": info.science_program,
                "jd": float(jd_start),
                "mjd": float(mjd_start),
            }
            rows.append(row)

        except ValueError as e:
            print(f"Erreur de valeur : {e}")
        except FileNotFoundError as e:
            print(f"Fichier introuvable : {e}")
        except Exception as e:
            print(f"Erreur inattendue : {type(e).__name__} - {e}")
            print(f">>>   Unexpected error at row {count}:", sys.exc_info()[0])
            traceback.print_exc()  # displays the full stack trace

    # Final DataFrame creation
    df_exposure = pd.DataFrame(rows)
    df_science = df_exposure[df_exposure.type == "science"]
    df_science.reset_index(drop=True, inplace=True)

    return df_science

In [ ]:
def FetchTimesForVisits(visit_list):
    """
    Keep for example and possible later use
    """
    # On interroge la table visitDefinition
    Nvisit = len(visit_list)

    if Nvisit == 1:
        thevisit = visit_list.values[0]
        rows = registry.queryDimensionRecords("visit", where=f"visit={thevisit}")
    else:
        rows = registry.queryDimensionRecords("visit", where=f"visit in {tuple(visit_list)}")

    # 4. Build a results table
    results = []
    for row in rows:
        visit_id = row.id
        visit_airmass = 1.0 / np.cos(Angle(row.zenith_angle, u.degree).rad)
        visit_azimuth = row.azimuth

        # Extraire l'instant de début de l'observation (Time astropy)
        start_time = row.timespan.begin

        # Convertir en MJD et ISO
        mjd = start_time.to_value("mjd")  # Ex: 60384.28718
        isot = start_time.to_value("isot")  # Ex: '2024-04-19 06:53:32.000'

        # mjd = row.startDate.toMjd()
        # utc = Time(mjd, format='mjd', scale='utc').to_value('iso')
        # results.append({"visit": visit_id, "mjd": mjd, "isot": isot})
        results.append(
            {"visit": visit_id, "mjd": mjd, "isot": isot, "airmass": visit_airmass, "azimuth": visit_azimuth}
        )

    df_times = pd.DataFrame(results).sort_values("visit")
    df_times.set_index("visit", inplace=True)
    return df_times

In [ ]:
def RetrieveDRPSources_forTarget(butler, center_coord, datasettype, where_clause, radius_cut=50):
    """
    Keep for example and possible later use
    Find the closest Sourcesthe target_coord

    parameters:
    - butler
    - the coordinate of the target (center of the cone seach)
    - the datasettype name for the DRP object
    - where_clause : which contrain requirements on the tract and patch numbers
    - cut on angluar separation for the returned for the returned object

    Return
    - object Id with minimum separation ,
    - minimum separation (arcec),
    - the table of DRP objects within the radius_cut
    """

    ra_columns = ["u_ra", "g_ra", "r_ra", "i_ra", "z_ra", "y_ra"]
    dec_columns = ["u_dec", "g_dec", "r_dec", "i_dec", "z_dec", "y_dec"]

    therefs = butler.registry.queryDatasets(datasettype, collections=collection, where=where_clause)

    for count, ref in enumerate(therefs):
        the_id = ref.dataId
        the_tract_id = the_id["tract"]
        print(the_id)

        # catalog of rubin objects (a pandas Dataframe) inside the tract
        catalog = butler.get(ref)
        catalog = catalog[catalog["patch"] == patchNbSel]

        nobjects = len(catalog)

        # Calcul de la moyenne ligne par ligne, en ignorant les NaN
        catalog["ra"] = catalog[ra_columns].mean(axis=1, skipna=True)
        catalog["dec"] = catalog[dec_columns].mean(axis=1, skipna=True)

        # extract the (ra,dec) coordinates for all te objects of the rubin-catalog
        ra_cat = catalog["ra"].values
        dec_cat = catalog["dec"].values
        # coordinates for all rubin-catalog points
        catalog_coords = SkyCoord(ra=ra_cat * u.deg, dec=dec_cat * u.deg)

        # Angular distance to target
        distances_arcsec = center_coord.separation(catalog_coords).arcsecond

        # add the separation angle to the ctalog
        catalog["sep"] = distances_arcsec

        # closest object from the target
        sepMin = distances_arcsec.min()
        sepMin_idx = np.where(distances_arcsec == sepMin)[0][0]

        closest_obj = catalog[catalog["sep"] <= sepMin]

        # select a few of these sources to debug the closest candidate
        nearby_obj = catalog[distances_arcsec < radius_cut]

        return closest_obj, sepMin, nearby_obj

In [ ]:
# =========================================================================
# Helper: add a top x-axis showing calendar dates above the MJD axis
# =========================================================================
# The bottom x-axis of each light curve panel uses MJD.
# This helper adds a twin axis on top with evenly-spaced date labels
# (UTC) so you can immediately read off known observing dates.
#
# Usage (call AFTER plotting, BEFORE tight_layout / savefig):
#   add_date_top_axis(axes[0], n_ticks=6)
# =========================================================================


def add_date_top_axis(ax, n_ticks=6, date_fmt="%Y-%m-%d"):
    """
    Add a secondary x-axis on top of `ax` showing calendar dates.

    The secondary axis shares the same MJD limits as the primary axis
    but labels ticks as UTC calendar dates (e.g. 2025-06-15).

    Parameters
    ----------
    ax : matplotlib Axes
        Primary axes whose bottom x-axis carries MJD values.
    n_ticks : int
        Number of evenly-spaced tick marks on the top axis.
    date_fmt : str
        strftime format for the date labels (default YYYY-MM-DD).

    Returns
    -------
    ax_top : matplotlib Axes
        The new twin axes placed on top.
    """
    mjd_min, mjd_max = ax.get_xlim()

    # Build evenly-spaced MJD tick positions spanning the current x range
    mjd_ticks = np.linspace(mjd_min, mjd_max, n_ticks)

    # Convert each MJD tick to a calendar date string via astropy
    date_labels = [Time(m, format="mjd", scale="utc").to_value("isot")[:10] for m in mjd_ticks]

    # Create a twin axis that shares the same x-scale
    ax_top = ax.twiny()
    ax_top.set_xlim(mjd_min, mjd_max)  # must match primary axis exactly
    ax_top.set_xticks(mjd_ticks)
    ax_top.set_xticklabels(date_labels, rotation=30, ha="left", fontsize=8)
    ax_top.set_xlabel("Date (UTC)", fontsize=9, labelpad=4)

    return ax_top


print("add_date_top_axis() helper defined.")

In [ ]:
def load_visits_file(file_path):
    """
    Checks if the visits file exists and loads it into a DataFrame.
    """
    if not os.path.exists(file_path):
        print(f"ERROR: File not found at {file_path}")
        return None

    try:
        # Standard LSST tables are usually Parquet
        if file_path.endswith(".parquet"):
            df_visits = pd.read_parquet(file_path)
        else:
            df_visits = pd.read_csv(file_path)

        print(f"Successfully loaded {len(df_visits):,} visits from {os.path.basename(file_path)}")
        return df_visits
    except Exception as e:
        print(f"ERROR loading file: {e}")
        return None


# --- Usage ---
# df_visits = load_visits_file(FULLFILENAME_VISITS)

---
## 1. User Configuration

In [ ]:
SELECTED_DDF = "ECDFS"  # COSMOS | ECDFS | ELAIS-S1 | XMM-LSS | EDFS-a | EDFS-b | EDFS | M49
# Check the selected TRACT and PATCH exist for the selected DDF
SELECTED_TRACT = 9813
# SELECTED_PATCH = 83
# SELECTED_PATCH = 58
CONE_RADIUS_DEG = 1.8
MIN_NSOURCES = 500

REPO = "dp2_prep"
COLLECTION = [
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage4",
]

SKYMAP_NAME = "lsst_cells_v2"
INSTRUMENT = "LSSTCam"
WHERE_CLAUSE_INSTRUMENT = "instrument = '" + f"{INSTRUMENT}" + "'"
DATE_START = 20250415
WHERE_CLAUSE_DATE = WHERE_CLAUSE_INSTRUMENT + f" and day_obs >= {DATE_START}"

TRACT_SEARCH_RADIUS_DEG = 1.8

DDF_COORDS = {
    "COSMOS": (150.119, +2.206),
    "ECDFS": (53.125, -28.100),
    "ELAIS-S1": (9.450, -44.000),
    "XMM-LSS": (35.708, -4.750),
    "EDFS-a": (58.900, -49.315),
    "EDFS-b": (63.600, -47.600),
    "EDFS": (61.240, -48.423),
    "M49": (187.400, +8.000),
}

RA_CENTER, DEC_CENTER = DDF_COORDS[SELECTED_DDF]
print(f"DDF         : {SELECTED_DDF}  ({RA_CENTER:.4f}°, {DEC_CENTER:+.4f}°)")
print(f"Cone radius : {CONE_RADIUS_DEG}°")
print(f"Min nSrc : {MIN_NSOURCES}")
print(f"Collection  : {COLLECTION}")
print(f"Instrument  : {INSTRUMENT}")
print(f"Where clause  : {WHERE_CLAUSE_INSTRUMENT}")
print(f"Where clause date : {WHERE_CLAUSE_DATE}")

---
## 2. Butler and SkyMap

In [ ]:
butler = Butler(REPO, collections=COLLECTION)
registry = butler.registry
print("Butler OK")

In [ ]:
try:
    skymap = butler.get("skyMap", skymap=SKYMAP_NAME, collections=COLLECTION)
except Exception:
    skymap = butler.get("skyMap", skymap=SKYMAP_NAME)
print(f"SkyMap '{SKYMAP_NAME}' : {len(skymap)} tracts")

### 2.1 Search for visit tables

In [ ]:
visitTable_pattern1 = "*isitTable*"
visitTable_pattern2 = "*isitTable"
visitTable_name = "preVisitTable"

In [ ]:
if DEBUG_VISITS:
    dataset_types = registry.queryDatasetTypes(visitTable_pattern1)
    for dt in dataset_types:
        print(dt.name)

In [ ]:
if DEBUG_VISITS:
    dataset_types = list(butler.registry.queryDatasetTypes(visitTable_pattern1))

    # Pour un affichage lisible (nom du type et storage class)
    for dt in sorted(dataset_types, key=lambda x: x.name):
        print(f"{dt.name:20} | {dt.storageClass.name}")

In [ ]:
if DEBUG_VISITS:
    dataset_types = list(butler.registry.queryDatasetTypes(visitTable_pattern2))
    for dt in dataset_types:
        print(f"{dt.name} :::", dt)
        print(f"    required dimensions: {dt.dimensions.required}")
        print()

## 2.2 Retrieve the science visits in order to match the visitid with the MJD in forced source photometry
- `visitid = id`
- `MJD = mjd `

In [ ]:
if FLAG_FETCH_VISITSEXPOSURES:
    visitTable = getLostOfVisits(registry, where_clause_date=WHERE_CLAUSE_DATE)
    print(visitTable.head(1))
    visitMin, visitMax, Nvisits = visitTable["id"].min(), visitTable["id"].max(), len(visitTable)
    print(f"visitMin = {visitMin},visitMax = {visitMax}, Nvisits = {Nvisits}")
    visit_filename = f"visitTable-{visitMin}-{visitMax}_N{Nvisits}.csv"
    print(f"filename_visits = {visit_filename}")
    visit_fullfilename = os.path.join(OUTPUT_DIR, visit_filename)
    visitTable.to_csv(visit_fullfilename)
    del visitTable
    gc.collect()

---
## 3. Dataset Type Inventory

Identify the correct names and **dimensions** for DIA products — in particular
`dia_object_forced_source` has `(skymap, tract, patch)` dimensions, which
requires iterating over patch refs.

In [ ]:
# Hard-coded after discovery: confirmed present in DM-54249
DATASET_TYPE_OBJ = "object"  # dims: skymap, tract
DATASET_TYPE_SRC = "source"  # dims: skymap, tract
DATASET_TYPE_FORCED = "object_forced_source"  # dims: skymap, tract, patch


# Print actual dimensions for confirmation
for dt_name in [DATASET_TYPE_OBJ, DATASET_TYPE_SRC, DATASET_TYPE_FORCED]:
    try:
        dt = registry.getDatasetType(dt_name)
        in_col = registry.queryDatasets(dt_name, collections=COLLECTION).any(execute=False, exact=False)
        print(f"{dt_name:40s}  dims={list(dt.dimensions.names)}  present={in_col}")
    except Exception as exc:
        print(f"{dt_name:40s}  ERROR: {exc}")

---
## 4. Find Tracts Covering the DDF

In [ ]:
def find_tracts_for_coord(skymap, ra_deg, dec_deg, radius_deg=1.8):
    cos_dec = max(np.cos(np.deg2rad(dec_deg)), 0.01)
    step = 0.35
    found_ids = set()
    for ddec in np.arange(-radius_deg, radius_deg + step, step):
        for dra in np.arange(-radius_deg, radius_deg + step, step):
            if np.sqrt(dra**2 + ddec**2) > radius_deg:
                continue
            ra_s = ra_deg + dra / cos_dec
            dec_s = dec_deg + ddec
            if not (-89.9 <= dec_s <= 89.9):
                continue
            sp = SpherePoint(ra_s * degrees, dec_s * degrees)
            try:
                found_ids.add(skymap.findTract(sp).tract_id)
            except Exception:
                pass
    return sorted(found_ids)


tract_ids = find_tracts_for_coord(skymap, RA_CENTER, DEC_CENTER, radius_deg=TRACT_SEARCH_RADIUS_DEG)
print(f"Tracts covering {SELECTED_DDF}: {tract_ids}")

---
## 5. Schema Discovery (one-time probe)

### 5.1 Object Schema

#### Probe DATASET_TYPE_OBJ 

In [ ]:
refs_probe = list(
    registry.queryDatasets(
        DATASET_TYPE_OBJ,
        collections=COLLECTION,
        dataId={"skymap": SKYMAP_NAME, "tract": tract_ids[0]},
    )
)
assert refs_probe, "No  object ref found."

obj_raw = butler.get(refs_probe[0])
df_probe = obj_raw.to_pandas() if hasattr(obj_raw, "to_pandas") else obj_raw

print(f"index.name  : {df_probe.index.name!r}")
print(f"index.dtype : {df_probe.index.dtype}")
print(f"index[:3]   : {df_probe.index[:3].tolist()}")
print(f"columns     : {list(df_probe.columns)[:50]}")

#### Probe OBJECT ID name

In [ ]:
# Determine ID column
if df_probe.index.name is not None:
    OBJ_ID_COL = df_probe.index.name
    ID_IN_INDEX = True
    print(f"Object ID in index: '{OBJ_ID_COL}'")
elif any(c in df_probe.columns for c in ["objectId", "object_id"]):
    OBJ_ID_COL = next(c for c in ["objectId", "object_id"] if c in df_probe.columns)
    ID_IN_INDEX = False
    print(f"Object ID as column: '{OBJ_ID_COL}'")
else:
    OBJ_ID_COL = "row_id"
    ID_IN_INDEX = False
    print("WARNING: no ID found, using row index.")

#### Probe existence of DATASET_TYPE_FORCED

In [ ]:
# Also probe object_forced_source schema (patch-level)
refs_forced_probe = list(
    registry.queryDatasets(
        DATASET_TYPE_FORCED,
        collections=COLLECTION,
        dataId={"skymap": SKYMAP_NAME, "tract": tract_ids[0]},
    )
)
print(f"object_forced_source refs for tract {tract_ids[0]}: {len(refs_forced_probe)}")

if refs_forced_probe:
    frc_raw = butler.get(refs_forced_probe[0])
    df_fprobe = frc_raw.to_pandas() if hasattr(frc_raw, "to_pandas") else frc_raw
    print(f"index.name  : {df_fprobe.index.name!r}")
    print(f"columns     : {list(df_fprobe.columns)}")
    print(f"shape       : {df_fprobe.shape}")

---
## 6. Helper: `to_dataframe()`

In [ ]:
def to_dataframe(obj, id_col=None, id_in_index=None):
    """
    Convert Butler catalog to pandas DataFrame.
    If id_in_index is True, promotes the named index to a regular column.
    Falls back to global OBJ_ID_COL / ID_IN_INDEX if not provided.
    """
    _id_col = id_col if id_col is not None else OBJ_ID_COL
    _id_idx = id_in_index if id_in_index is not None else ID_IN_INDEX

    if isinstance(obj, pd.DataFrame):
        df = obj
    elif hasattr(obj, "to_pandas"):
        df = obj.to_pandas()
    elif hasattr(obj, "to_table"):
        df = obj.to_table().to_pandas()
    elif isinstance(obj, Table):
        df = obj.to_pandas()
    else:
        raise TypeError(f"Cannot convert {type(obj)} to DataFrame.")

    if _id_idx and _id_col not in df.columns:
        df = df.copy()
        df.insert(0, _id_col, df.index)
        df = df.reset_index(drop=True)
    elif _id_col == "row_id" and _id_col not in df.columns:
        df.insert(0, _id_col, range(len(df)))

    return df

---
## 7. Load Object Catalog

### 7.1 Select only one TRACT

In [ ]:
# tract_ids = [SELECTED_TRACT]

### 7.2 Retrieve Objects from the Butler in order to build the dict of tract-patches

In [ ]:
# loop on tracts
tractpatchDict = {}
for tract_id in tract_ids:
    refs = list(
        registry.queryDatasets(
            DATASET_TYPE_OBJ, collections=COLLECTION, dataId={"skymap": SKYMAP_NAME, "tract": tract_id}
        )
    )
    for ref in refs:
        try:
            df_tmp = to_dataframe(butler.get(ref))
            df_tmp["_tract"] = tract_id

            list_of_patches = df_tmp["patch"].unique()
            # print(f"tract : {tract_id} ==> list_of_patches : ", list_of_patches)

            # Calcul du nombre d'objets par patch
            # summary = df_tmp.groupby('patch').size()
            # summary = df_tmp.groupby('patch').agg(patchcount=('patch', 'size')).reset_index()
            summary = df_tmp.groupby("patch").size().rename("patchcount")
            # summary = df_tmp.groupby('patch').size().to_frame("patchcount")
            total_objects = len(df_tmp)
            num_patches = len(summary)

            tractpatchDict[tract_id] = summary

            # clean memory from objects table
            del df_tmp
            gc.collect()

            print(f"  Tract {tract_id:6d} — {total_objects:,} Objects")
        except Exception as exc:
            print(f"  Tract {tract_id:6d} — ERROR: {exc}")

### 7.3 Convert the tract-patch dictionnary to assess object density in tract-patches 

In [ ]:
df_plot = pd.concat(tractpatchDict, names=["tract", "patch"]).reset_index()

In [ ]:
df_plot.head()

In [ ]:
pivot = df_plot.pivot(index="tract", columns="patch", values="patchcount")

plt.figure(figsize=(18, 8))
im = plt.imshow(pivot, aspect="auto")

plt.colorbar(im, label="patchcount")

# 🔑 Ajout des labels
plt.yticks(ticks=range(len(pivot.index)), labels=pivot.index)

plt.xticks(ticks=range(len(pivot.columns)), labels=pivot.columns, rotation=90)

plt.xlabel("Patch")
plt.ylabel("Tract")
plt.title(f"Patchcount par tract for selected DDF {SELECTED_DDF}")

plt.tight_layout()
plt.show()

In [ ]:
for tract, group in df_plot.groupby("tract"):
    plt.figure(figsize=(18, 3))
    plt.bar(group["patch"].astype(str), group["patchcount"])
    plt.xticks(rotation=90)
    plt.title(f"Tract {tract} : for selected DDF {SELECTED_DDF}")
    plt.xlabel("Patch")
    plt.ylabel("Patchcount")
    plt.tight_layout()
    plt.grid()
    plt.show()

## Select tract patch

In [ ]:
TRACT_SELECTED = 9813
PATCH_SELECTED = 50

In [ ]:
# Types of datasets searched
dataset_coadd = "deepCoadd"
dataset_dia = "goodSeeingDiff_differenceExp"

In [ ]:
assert False

### Search for datsets in the collection

```
Analyse de la hiérarchie pour LSSTCam/runs/DRP/20250417_20250921/w_2025_49/DM-53545...
✅ TROUVÉ : 'direct_warp'
   DANS : LSSTCam/runs/DRP/20250417_20250921/w_2025_49/DM-53545/20251211T040218Z
   EXEMPLE ID : {instrument: 'LSSTCam', skymap: 'lsst_cells_v2', tract: 10054, patch: 31, visit: 2025051900222, band: 'y', day_obs: 20250519, physical_filter: 'y_10'}
✅ TROUVÉ : 'direct_warp_masked_fraction'
   DANS : LSSTCam/runs/DRP/20250417_20250921/w_2025_49/DM-53545/20251211T040218Z
   EXEMPLE ID : {instrument: 'LSSTCam', skymap: 'lsst_cells_v2', tract: 10054, patch: 31, visit: 2025051900222, band: 'y', day_obs: 20250519, physical_filter: 'y_10'}
✅ TROUVÉ : 'makeDirectWarp_log'
   DANS : LSSTCam/runs/DRP/20250417_20250921/w_2025_49/DM-53545/20251211T040218Z
   EXEMPLE ID : {instrument: 'LSSTCam', skymap: 'lsst_cells_v2', tract: 9812, patch: 75, visit: 2025052300094, band: 'u', day_obs: 20250523, physical_filter: 'u_24'}
✅ TROUVÉ : 'makeDirectWarp_metadata'
   DANS : LSSTCam/runs/DRP/20250417_20250921/w_2025_49/DM-53545/20251211T040218Z
   EXEMPLE ID : {instrument: 'LSSTCam', skymap: 'lsst_cells_v2', tract: 3725, patch: 52, visit: 2025052000310, band: 'i', day_obs: 20250520, physical_filter: 'i_39'}
✅ TROUVÉ : 'makePsfMatchedWarp_log'
   DANS : LSSTCam/runs/DRP/20250417_20250921/w_2025_49/DM-53545/20251211T040218Z
   EXEMPLE ID : {instrument: 'LSSTCam', skymap: 'lsst_cells_v2', tract: 2703, patch: 72, visit: 2025071900606, band: 'u', day_obs: 20250719, physical_filter: 'u_24'}
```

In [ ]:
if 0:
    chain = butler.registry.getCollectionChain(COLLECTION)
    found_image_types = set()

    print(f"Analyse de la hiérarchie pour {COLLECTION}...")

    # 1. We retrieve the list of ALL possible dataset types
    all_ds_types = list(butler.registry.queryDatasetTypes())

    for ds_type in all_ds_types:
        dims = ds_type.dimensions.names
        # Une image de coadd doit avoir au moins 'patch', 'tract' et 'band'
        if "patch" in dims and "band" in dims and "tract" in dims:
            # 2. On teste chaque collection de la chaîne pour ce type
            for coll in chain:
                try:
                    # We retrieve the results. Since we cannot use 'limit',
                    # we convert the iterator to a list and check if it is not empty.
                    refs = list(butler.registry.queryDatasets(ds_type.name, collections=coll))

                    if len(refs) > 0:
                        print(f"✅ TROUVÉ : '{ds_type.name}'")
                        print(f"   DANS : {coll}")
                        print(f"   EXEMPLE ID : {refs[0].dataId}")
                        found_image_types.add((ds_type.name, coll))
                        break  # Found for this type, we move to the next DatasetType
                except Exception:
                    continue

    if not found_image_types:
        print("\n❌ Aucun dataset d'image (avec patch/band) n'a été trouvé.")
        print("Vérifions les types disponibles sans filtre :")
        # Affiche les 10 premiers types juste pour voir à quoi ils ressemblent
        sample = [dt.name for dt in all_ds_types[:10]]
        print(f"Exemples de types existants : {sample}")

In [ ]:
fc = FireflyClient.make_client(launch_browser=False)
# Check if this works by requesting the URL
print(f"Firefly est disponible ici : {fc.get_firefly_url()}")

In [ ]:
# On prend l'ID de votre exemple
warp_id = {
    "instrument": "LSSTCam",
    "skymap": "lsst_cells_v2",
    "tract": 10054,
    "patch": 31,
    "visit": 2025051900222,
    "band": "y",
}
warp_exp = butler.get("direct_warp", dataId=warp_id)
afwDisplay.Display(frame=1).mtv(warp_exp)

In [ ]:
assert False

In [ ]:
# Test d'affichage simple
dataId = {
    "skymap": SKYMAP_NAME,
    "tract": TRACT_SELECTED,
    "patch": PATCH_SELECTED,
    "band": "g",
}  # Remplacez par un patch valide
exp = butler.get("deepCoadd", dataId=dataId)

# Utilisez le viewer par défaut
fc.show_lsst_exposure(exp)

In [ ]:
BANDS = ["u", "g", "r", "i", "z", "y"]

In [ ]:
refs = butler.registry.queryDatasets(
    "deepCoadd", collections=COLLECTION, dataId={"skymap": SKYMAP_NAME, "tract": SELECTED_TRACT}
)

In [ ]:
# On extrait le patch de chaque référence trouvée
patch_ids = sorted(list(set(ref.dataId["patch"] for ref in refs)))

print(f"Patchs trouvés via queryDatasets : {patch_ids}")

In [ ]:
BANDS = ["u", "g", "r", "i", "z", "y"]

# --- ÉTAPE 1 : Lister les patchs disponibles pour ce tract ---
# On utilise queryDataIds pour obtenir les combinaisons valides de patchs
patch_ids = sorted(
    butler.registry.queryDataIds(
        ["patch"],
        datasets="deepCoadd",
        collections=COLLECTION,
        dataId={"skymap": SKYMAP_NAME, "tract": SELECTED_TRACT},
    )
    .to_pandas()["patch"]
    .unique()
)

print(f"Patchs disponibles pour le tract {SELECTED_TRACT}: {patch_ids}")

# --- ÉTAPE 2 : Visualisation du patch choisi ---
target_patch = patch_ids[0]  # On prend le premier pour l'exemple

for band in BANDS:
    dataId = {"skymap": SKYMAP_NAME, "tract": SELECTED_TRACT, "patch": target_patch, "band": band}

    try:
        # Récupération de l'image
        exposure = butler.get("deepCoadd", dataId=dataId)

        # Nom unique pour le viewer Firefly (un par bande ou un seul pour tout)
        # Si vous voulez tout voir en même temps, utilisez un ID différent par bande
        viewer_id = f"view_{band}"

        title_str = f"Tract: {SELECTED_TRACT} | Patch: {target_patch} | Band: {band}"

        # Affichage
        fc.show_lsst_exposure(exposure, viewer_id=viewer_id, title=title_str)

        # Grille RA/Dec
        fc.set_grid_state(True, viewer_id=viewer_id)

        print(f"Bande {band} envoyée à Firefly.")

    except Exception as e:
        print(f"Bande {band} non trouvée ou erreur : {e}")

# Synchronisation des vues (Zoom/Pan) si vous avez plusieurs fenêtres
fc.set_sync_view(True)

In [ ]:
ds_refs = butler.registry.queryDatasets(dataset_coadd, collections=COLLECTION)

# Extraction des tracts et patchs uniques
available_data = []
for ref in ds_refs:
    available_data.append(
        {"tract": ref.dataId["tract"], "patch": ref.dataId["patch"], "band": ref.dataId["band"]}
    )

# Exemple : Lister les tracts disponibles
unique_tracts = sorted(list(set(d["tract"] for d in available_data)))
print(f"Tracts disponibles : {unique_tracts}")